# Pemodelan & Simulasi Dinamik: Konsentrasi Paracetamol dalam Darah
Model farmakokinetik satu kompartemen dengan absorpsi oral.

**Nama:** ........  **NIM:** ........

> Model ini untuk pembelajaran pemodelan sistem dinamik, bukan panduan dosis obat.

## 1. Import library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

## 2. Parameter model
$F$ bioavailabilitas, $k_a$ laju absorpsi, $k_e$ laju eliminasi, $V_d$ volume distribusi.

In [ ]:
D   = 1000.0   # dosis per minum (mg) = 2 tablet 500 mg
F   = 0.85     # bioavailabilitas (fraksi yang terserap)
ka  = 1.5      # laju absorpsi (1/jam)
ke  = 0.35     # laju eliminasi (1/jam), waktu paruh ~ 2 jam
Vd  = 50.0     # volume distribusi (L)

## 3. Model ODE
$$\frac{dG}{dt}=-k_aG,\qquad \frac{dC}{dt}=\frac{k_aG}{V_d}-k_eC$$

In [ ]:
# x[0] = G (massa obat di lambung/usus, mg)
# x[1] = C (konsentrasi obat dalam darah, mg/L)
def obat(x, t, ka, ke, Vd):
    G, C = x
    dGdt = -ka * G
    dCdt = (ka * G) / Vd - ke * C
    return [dGdt, dCdt]

## 4. Simulasi 1: dosis tunggal vs solusi analitik

In [ ]:
dt = 0.1
ts = np.arange(0, 24 + dt, dt)
x0 = [F * D, 0.0]
x1 = odeint(obat, x0, ts, args=(ka, ke, Vd))

# solusi analitik (pembanding)
C_an = (F * D * ka) / (Vd * (ka - ke)) * (np.exp(-ke * ts) - np.exp(-ka * ts))


plt.figure(figsize=(8,4))
plt.plot(ts, x1[:,1],'b-',label='Numerik (odeint)')
plt.plot(ts, C_an,'r--',label='Analitik')
plt.xlabel('Waktu (jam)'); plt.ylabel('C(t) (mg/L)'); plt.legend(); plt.grid(alpha=.3); plt.show()
print('Cmax = %.3f mg/L' % x1[:,1].max())

## 5. Simulasi 2: dosis berulang tiap 6 jam

In [ ]:
jadwal = [0, 6, 12, 18]            # jam minum obat
G_all = np.zeros(len(ts))
C_all = np.zeros(len(ts))
x = [0.0, 0.0]
for i in range(len(ts) - 1):
    if round(ts[i], 1) in jadwal:   # saat minum obat, massa di lambung bertambah
        x[0] += F * D
    if i == 0:
        G_all[0], C_all[0] = x[0], x[1]
    y = odeint(obat, x, [ts[i], ts[i + 1]], args=(ka, ke, Vd))
    x = list(y[-1])
    G_all[i + 1], C_all[i + 1] = x


plt.figure(figsize=(8,4))
plt.plot(ts, C_all,'b-')
plt.axhline(10,color='orange',ls='--',label='Batas bawah efektif (asumsi)')
plt.axhline(20,color='red',ls='--',label='Batas atas aman (asumsi)')
plt.xlabel('Waktu (jam)'); plt.ylabel('C(t) (mg/L)'); plt.legend(frameon=False); plt.grid(alpha=.3); plt.show()

## 6. Simulasi 3: variasi laju eliminasi $k_e$

In [ ]:
plt.figure(figsize=(8, 4))
for k_e, gaya in [(0.2, 'r-'), (0.35, 'b--'), (0.6, 'g:')]:
    y = odeint(obat, x0, ts, args=(ka, k_e, Vd))
    plt.plot(ts, y[:, 1], gaya, label=f'$k_e$ = {k_e}')
plt.xlabel('Waktu (jam)'); plt.ylabel('C(t) (mg/L)')
plt.title('Pengaruh laju eliminasi $k_e$ (dosis tunggal)')
plt.legend(); plt.grid(alpha=.3); plt.show()

# ---------- Plot 1 ----------
fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax[0].plot(ts, x1[:, 1], 'b-', label='Numerik (odeint)')
ax[0].plot(ts, C_an, 'r--', label='Analitik')
ax[0].set_ylabel('C(t) (mg/L)'); ax[0].legend(frameon=False); ax[0].grid(alpha=.3)
ax[0].set_title('Dosis tunggal 1000 mg')
ax[1].plot(ts, x1[:, 0], 'g-')
ax[1].set_ylabel('G(t) (mg)'); ax[1].set_xlabel('Waktu (jam)'); ax[1].grid(alpha=.3)
plt.savefig('grafik1_dosis_tunggal.png', dpi=150, bbox_inches='tight'); plt.close()

# ---------- Plot 2 ----------
fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax[0].plot(ts, C_all, 'b-')
ax[0].axhline(10, color='orange', ls='--', label='Batas bawah efektif (10 mg/L, asumsi)')
ax[0].axhline(20, color='red', ls='--', label='Batas atas aman (20 mg/L, asumsi)')
ax[0].set_ylabel('C(t) (mg/L)'); ax[0].legend(frameon=False, fontsize=8); ax[0].grid(alpha=.3)
ax[0].set_title('Dosis 1000 mg setiap 6 jam')
u = np.zeros(len(ts))
for j in jadwal: u[int(round(j / dt))] = D
ax[1].stem(ts[u > 0], u[u > 0], linefmt='r-', markerfmt='ro', basefmt=' ')
ax[1].set_ylabel('Dosis u(t) (mg)'); ax[1].set_xlabel('Waktu (jam)'); ax[1].grid(alpha=.3)
plt.savefig('grafik2_dosis_berulang.png', dpi=150, bbox_inches='tight'); plt.close()

## 7. Perhitungan manual (metode Euler, h = 0,25 jam)

In [ ]:
h = 0.25
G, C = F * D, 0.0
rows = []
for n in range(0, 13):
    t = n * h
    Cana = (F*D*ka)/(Vd*(ka-ke))*(np.exp(-ke*t)-np.exp(-ka*t))
    rows.append((t, G, C, Cana))
    G, C = G + h*(-ka*G), C + h*((ka*G)/Vd - ke*C)


print('t | G | C(Euler) | C(analitik)')
for r in rows: print('%.2f | %.2f | %.3f | %.3f' % r)

## 8. (Opsional) Solusi dengan GEKKO

In [ ]:
# OPSIONAL: solusi dengan GEKKO (dosis tunggal, bandingkan dengan odeint)
!pip install gekko -q
from gekko import GEKKO

m = GEKKO()
m.time = np.linspace(0, 24, 241)
G = m.Var(F * D)          # massa obat di lambung (mg)
C = m.Var(0.0)            # konsentrasi di darah (mg/L)
m.Equation(G.dt() == -ka * G)
m.Equation(C.dt() == (ka * G) / Vd - ke * C)
m.options.IMODE = 4       # simulasi dinamik
m.solve(disp=False)

plt.plot(m.time, C, 'b-', label='GEKKO')
plt.plot(ts, x1[:, 1], 'r--', label='odeint')
plt.xlabel('Waktu (jam)'); plt.ylabel('C(t) (mg/L)'); plt.legend(); plt.grid(alpha=.3)
plt.show()